# Nail Retouch Masked Inpaint Full12 v1

This notebook rebuilds the full approved 12-sample explicit masked dataset and trains the masked inpainting LoRA route on Colab.

It follows the same structure as the historical paired-edit notebooks: mount Drive, load a YAML config, prepare local data, launch training, and sync artifacts back to Drive.

## 1. Clone Project

Use this in a fresh Colab runtime:
```bash
cd /content && rm -rf nail-retouch-assistant && git clone -b codex/masked-full12-colab https://github.com/Zifpen/nail-retouch-assistant.git
```

Expected Drive paths:
- Raw pairs under `/content/drive/MyDrive/nail-retouch-raw/pair_XXXX/`
- Optional cached masked dataset under `/content/drive/MyDrive/masked_inpaint_cuticle_cleanup_v1_full12/`
- Training outputs under `/content/drive/MyDrive/nail-retouch-masked-full12-outputs/`

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
import yaml

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/nail-retouch-assistant')
CONFIG_FILE = 'masked_inpaint_full12_config.yaml'
CONFIG_PATH = PROJECT_ROOT / 'colab' / CONFIG_FILE
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))

if not PROJECT_ROOT.exists():
    raise FileNotFoundError('Project repo is missing. Run the clone command in the markdown cell first.')

subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'colab' / 'requirements.txt')], check=True)
subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', 'diffusers', 'transformers', 'accelerate', 'peft', 'sentencepiece', 'safetensors'],
    check=True,
)
print('Setup complete')

In [ ]:
from pathlib import Path
import shutil
import subprocess
import yaml

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
PROJECT_ROOT = Path(cfg['project_root'])
LOCAL_DATASET_DIR = PROJECT_ROOT / cfg['dataset_dir']
MANIFEST_PATH = PROJECT_ROOT / cfg['manifest_path']
RAW_DIR = PROJECT_ROOT / cfg['raw_dir']
DRIVE_RAW_DIR = Path(cfg['drive_raw_dir'])
DRIVE_DATASET_DIR = Path(cfg['drive_dataset_dir'])

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Missing manifest: {MANIFEST_PATH}')

shutil.rmtree(LOCAL_DATASET_DIR, ignore_errors=True)
LOCAL_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

if cfg.get('rebuild_dataset', True):
    if not DRIVE_RAW_DIR.exists():
        raise FileNotFoundError(f'Missing raw pair root in Drive: {DRIVE_RAW_DIR}')
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    RAW_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        f"cp -a '{DRIVE_RAW_DIR}' '{RAW_DIR}'",
        shell=True,
        check=True,
        executable='/bin/bash',
    )
    build_cmd = [
        'python3',
        str(PROJECT_ROOT / 'src' / 'data' / 'build_masked_inpaint_dataset.py'),
        '--raw-dir',
        str(RAW_DIR),
        '--manifest',
        str(MANIFEST_PATH),
        '--output-dir',
        str(LOCAL_DATASET_DIR),
        '--mask-mode',
        'explicit',
    ]
    subprocess.run(build_cmd, check=True)
    if cfg.get('sync_built_dataset_to_drive', False):
        shutil.rmtree(DRIVE_DATASET_DIR, ignore_errors=True)
        DRIVE_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            f"cp -a '{LOCAL_DATASET_DIR}' '{DRIVE_DATASET_DIR}'",
            shell=True,
            check=True,
            executable='/bin/bash',
        )
else:
    if not DRIVE_DATASET_DIR.exists():
        raise FileNotFoundError(f'Missing prepared dataset in Drive: {DRIVE_DATASET_DIR}')
    subprocess.run(
        f"cp -a '{DRIVE_DATASET_DIR}' '{LOCAL_DATASET_DIR}'",
        shell=True,
        check=True,
        executable='/bin/bash',
    )

print(f'Dataset ready at {LOCAL_DATASET_DIR}')

In [ ]:
from pathlib import Path
import json
import torch
import yaml

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
dataset_dir = Path(cfg['project_root']) / cfg['dataset_dir']
output_dir = Path(cfg['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. Switch Colab runtime to T4/L4/A100 before training.')
print('GPU:', torch.cuda.get_device_name(0))
print('dataset:', dataset_dir)
print('config:', CONFIG_PATH)
cfg

In [ ]:
import json
from pathlib import Path

def load_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

train_rows = load_jsonl(dataset_dir / 'train' / 'metadata.jsonl')
val_rows = load_jsonl(dataset_dir / 'val' / 'metadata.jsonl')
print('train pairs:', len(train_rows))
print('val pairs:', len(val_rows))
print('example train pair:', train_rows[0]['id'])
print('example prompt:', train_rows[0]['prompt'])

In [ ]:
import shlex
from pathlib import Path

train_cfg = cfg['training']
cmd_parts = [
    'python3',
    shlex.quote(str(PROJECT_ROOT / 'src' / 'training' / 'train_masked_inpaint_lora.py')),
    '--pretrained_model_name_or_path', shlex.quote(cfg['pretrained_model_name_or_path']),
    '--dataset_dir', shlex.quote(str(dataset_dir)),
    '--output_dir', shlex.quote(cfg['output_dir']),
    '--resolution', str(train_cfg['resolution']),
    '--image_prep', shlex.quote(train_cfg.get('image_prep', 'resize')),
    '--train_batch_size', str(train_cfg['train_batch_size']),
    '--gradient_accumulation_steps', str(train_cfg.get('gradient_accumulation_steps', 1)),
    '--num_train_epochs', str(train_cfg['num_train_epochs']),
    '--max_train_steps', str(train_cfg['max_train_steps']),
    '--checkpointing_steps', str(train_cfg['checkpointing_steps']),
    '--preview_steps', str(train_cfg['preview_steps']),
    '--validation_samples', str(train_cfg.get('validation_samples', 4)),
    '--max_preview_images', str(train_cfg.get('max_preview_images', 4)),
    '--dataloader_num_workers', str(train_cfg.get('dataloader_num_workers', 0)),
    '--learning_rate', str(train_cfg['learning_rate']),
    '--lr_scheduler', shlex.quote(train_cfg.get('lr_scheduler', 'constant')),
    '--lr_warmup_steps', str(train_cfg.get('lr_warmup_steps', 0)),
    '--rank', str(train_cfg.get('rank', 4)),
    '--lora_dropout', str(train_cfg.get('lora_dropout', 0.05)),
    '--mixed_precision', shlex.quote(train_cfg.get('mixed_precision', 'fp16')),
    '--lambda_diffusion', str(train_cfg.get('lambda_diffusion', 1.0)),
    '--lambda_mask_rgb', str(train_cfg.get('lambda_mask_rgb', 1.0)),
    '--lambda_identity', str(train_cfg.get('lambda_identity', 5.0)),
    '--lambda_color', str(train_cfg.get('lambda_color', 0.5)),
    '--mask_dilate', str(train_cfg.get('mask_dilate', 3)),
]

if train_cfg.get('gradient_checkpointing', False):
    cmd_parts.append('--gradient_checkpointing')
if train_cfg.get('allow_tf32', False):
    cmd_parts.append('--allow_tf32')
if train_cfg.get('enable_xformers_memory_efficient_attention', False):
    cmd_parts.append('--enable_xformers_memory_efficient_attention')
if train_cfg.get('allow_dataset_prompt_variants', False):
    cmd_parts.append('--allow_dataset_prompt_variants')

cmd = ' '.join(cmd_parts)
print(cmd)

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import threading

train_cfg = cfg['training']
DRIVE_OUTPUT_DIR = Path(cfg['drive_output_dir'])
LOCAL_OUTPUT_DIR = Path(cfg['output_dir'])
SYNC_INTERVAL_SECONDS = int(train_cfg.get('sync_to_drive_interval_seconds', 120))

def copy_tree_contents(src: Path, dst: Path) -> None:
    dst.mkdir(parents=True, exist_ok=True)
    for child in src.iterdir():
        target = dst / child.name
        if child.is_dir():
            copy_tree_contents(child, target)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(child, target)

def sync_outputs(label: str = 'Sync') -> None:
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if not LOCAL_OUTPUT_DIR.exists():
        return
    for item in LOCAL_OUTPUT_DIR.iterdir():
        dst = DRIVE_OUTPUT_DIR / item.name
        try:
            if item.is_dir():
                copy_tree_contents(item, dst)
            else:
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(item, dst)
        except Exception as exc:
            print(f'{label} warning for {item.name}: {exc}')
    print(f'{label}: copied outputs into {DRIVE_OUTPUT_DIR} without deleting existing files')

env = os.environ.copy()
env['WANDB_MODE'] = 'disabled'
env['WANDB_DISABLED'] = 'true'

sync_stop = threading.Event()

def sync_worker() -> None:
    while not sync_stop.wait(SYNC_INTERVAL_SECONDS):
        sync_outputs(label='Periodic sync')

sync_outputs(label='Initial sync')
sync_thread = threading.Thread(target=sync_worker, daemon=True)
sync_thread.start()

process = subprocess.Popen(
    cmd,
    shell=True,
    executable='/bin/bash',
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end='')
finally:
    process.wait()
    sync_stop.set()
    sync_thread.join(timeout=5)
    sync_outputs(label='Final sync')

print('\nRETURN CODE:', process.returncode)
if process.returncode != 0:
    raise RuntimeError(f'Masked training failed with exit code {process.returncode}')

In [ ]:
from pathlib import Path
from IPython.display import Image, display

preview_dir = Path(cfg['drive_output_dir']) / 'previews'
checkpoint_dir = Path(cfg['drive_output_dir']) / 'lora_checkpoints'

print('drive output dir:', Path(cfg['drive_output_dir']))
print('checkpoints:')
for path in sorted(checkpoint_dir.glob('*')):
    print(path)

preview_candidates = sorted(preview_dir.glob('preview_step_*.png'))
if preview_candidates:
    print('latest preview:', preview_candidates[-1])
    display(Image(filename=str(preview_candidates[-1])))
else:
    print('No preview written yet')

## Notes

- This notebook rebuilds the approved explicit masked dataset from Drive raw pairs by default so the dataset and manifest stay coupled to the checked-in repo state.
- The current config is tuned for a first meaningful GPU-side masked run, not for ultra-fast smoke checks.
- Keep the dataset fixed while testing longer run lengths so result interpretation stays clean.